# Mini Project - CNN (Covid-19 Image Detection)

In [ ]:
# RUN THIS CELL ONLY WHEN IMPORTING ANY DATASET FROM KAGGLE INTO GOOGLE COLAB.

# Mounting Google Drive (tiwari.21@iitj.ac.in account) into Google Colab.
from google.colab import drive
drive.mount("/content/drive")

# Connecting Kaggle with Google Colab.
!mkdir -p ~/.kaggle # Creates the hidden .kaggle folder in our home directory. -p prevents errors if it already exists.
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ # Copies kaggle.json directly from my Google Drive into the .kaggle folder — instead of manually uploading it every session.
!chmod 600 ~/.kaggle/kaggle.json # Locks the file so only I can read/write it — required by Kaggle's API client.

In [ ]:
# Downloading the required dataset.
!kaggle datasets download -d pranavraikokte/covid19-image-dataset

In [ ]:
# Unzipping the zipped file.
import zipfile

zip_ref = zipfile.ZipFile("/content/covid19-image-dataset.zip", "r") # "r" means in read-only mode.
zip_ref.extractall("/content/covid19")
zip_ref.close()

In [ ]:
# Importing the required library.
!pip install kaggle keras-tuner -q # q = quiet mode.

In [ ]:
# Importing other libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, cv2
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import keras_tuner as kt

In [ ]:
# Checking the structure of the dataset.
for root, dirs, files in os.walk("covid19"): # os.walk() is a Python function that recursively traverses a directory tree, yielding the folder path, subfolders, and files at each level.
  print(root, "=", len(files), "files")

In [ ]:
# Loading images.
IMG_SIZE = 128
DATA_PATH = "/content/covid19/Covid19-dataset/train"
CLASSES = ["Covid", "Normal", "Viral Pneumonia"]

X = []
y = []

for label in CLASSES:
  path = os.path.join(DATA_PATH, label)
  for img in os.listdir(path):
    img_path = os.path.join(path, img)
    img = cv2.imread(img_path) # cv2.imread(img_path) loads the image as a NumPy array in BGR (Blue, Green, Red) format (not RGB) — OpenCV's (Open Source Computer Vision Library) default color channel order). Returns None if the file can't be read (which is why our code checks if img is not None).
    if img is not None:
      img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) # Note OpenCV takes (width, height), not (height, width).
      X.append(img)
      y.append(label)

X = np.array(X)
y = np.array(y)
print(f"Dataset Shape: {X.shape}, Labels: {np.unique(y)}")

##### 251 — total images loaded across all 3 classes
##### 128 — image height (pixels)
##### 128 — image width (pixels)
##### 3 — color channels (Red, Green, Blue)

In [ ]:
# Preprocessing.
# Normalizing.
X = X / 255.0 # Normalizes the image pixel values from the range [0, 255] to [0.0, 1.0]. This is because smaller numbers train faster and provides stable gradient updates.

# Encoding the labels to integers and then to vectors.
le = LabelEncoder()
y_encoded = le.fit_transform(y) # y_encoded returns [0, 1, 2] integers for the 3 classes/labels ["Covid", "Normal", "Viral Pneumonia"] respectively.
y_categorical = to_categorical(y_encoded, num_classes = 3) # y_categorical returns [1, 0, 0], [0, 1, 0], and [0, 0, 1] vectors for the 3 integers [0, 1, 2].
NUM_CLASSES = 3

In [ ]:
# Train/Val/Test split (70/15/15).
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size = 0.50, random_state = 42, stratify = y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.50, random_state = 42)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Validation: {X_val.shape}, {y_val.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

In [ ]:
# Class weights for imbalance.
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(class_weight = "balanced", classes = np.unique(y_encoded), y = y_encoded)
class_weights_dict = dict(enumerate(class_weights))
print("Class Weights: ", class_weights_dict)

In [ ]:
# Callbacks.
callbacks = [
    EarlyStopping(patience = 5, restore_best_weights = True, verbose = 1),
    ReduceLROnPlateau(patience = 3, factor = 0.5, verbose = 1)
]

In [ ]:
# Model 1: Basic CNN.
# Building the basic CNN model.
model1 = Sequential([
    Conv2D(32, (3, 3), activation = "relu", input_shape = (IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation = "relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation = "relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation = "softmax")
])

In [ ]:
# Compiling the basic CNN model.
model1.compile(optimizer = "adam", loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
model1.summary()

In [ ]:
# Training the basic CNN model.
history1 = model1.fit(X_train, y_train,
                     validation_data = (X_val, y_val), epochs = 20, batch_size = 32,
                     class_weight = class_weights_dict, callbacks = callbacks)

In [ ]:
# Model 2: Transfer Learning (Using pre-trained model VGG16).
base_vgg = VGG16(weights = "imagenet", include_top = False, input_shape = (IMG_SIZE, IMG_SIZE, 3))
base_vgg.trainable = False # Freeze base.

x = GlobalAveragePooling2D()(base_vgg.output)
x = Dense(256, activation = "relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation = "softmax")(x)

In [ ]:
# Compiling model 2 (VGG16).
model2 = Model(inputs = base_vgg.input, outputs = output)
model2.compile(optimizer = "adam", loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
model2.summary()

In [ ]:
# Training model 2 (VGG16).
history2 = model2.fit(X_train, y_train,
                     validation_data = (X_val, y_val), epochs = 20, batch_size = 32,
                     class_weight = class_weights_dict, callbacks = callbacks)

In [ ]:
# Model 3: Transfer Learning + Data Augmentation (Using pre-trained model ResNet50).
datagen = ImageDataGenerator(
    rotation_range = 15, width_shift_range = 0.1, height_shift_range = 0.1,
    zoom_range = 0.2, horizontal_flip = True)

datagen.fit(X_train)

base_resnet = ResNet50(weights = "imagenet", include_top = False, input_shape = (IMG_SIZE, IMG_SIZE, 3))
base_resnet.trainable = False # Freeze base.

x = GlobalAveragePooling2D()(base_resnet.output)
x = Dense(256, activation = "relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation = "softmax")(x)

In [ ]:
# Compiling model 3 (ResNet50).
model3 = Model(inputs = base_resnet.input, outputs = output)
model3.compile(optimizer = "adam", loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
model3.summary()

In [ ]:
# Training model 3 (ResNet50).
history3 = model3.fit(datagen.flow(X_train, y_train, batch_size = 32),
                      validation_data = (X_val, y_val), epochs = 20,
                      class_weight = class_weights_dict, callbacks = callbacks)

In [ ]:
def evaluate_model(model, history, name):
    fig, axes = plt.subplots(1, 2, figsize = (12, 4))

    axes[0].plot(history.history["accuracy"], label = "Train")
    axes[0].plot(history.history["val_accuracy"], label = "Val")
    axes[0].set_title(f"{name} - Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history.history["loss"], label = "Train")
    axes[1].plot(history.history["val_loss"], label = "Val")
    axes[1].set_title(f"{name} - Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis = 1)
    y_true = np.argmax(y_test, axis = 1)
    print(f"\n-{name}-")

    print(classification_report(y_true, y_pred, target_names = CLASSES))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize = (6, 4))
    sns.heatmap(cm, annot = True, fmt = "d", cmap = "Blues",
                xticklabels = CLASSES, yticklabels = CLASSES)
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    train_accuracy = max(history.history["accuracy"])
    test_accuracy = model.evaluate(X_test, y_test, verbose = 0)[1]
    overfit = "Yes" if (train_accuracy - test_accuracy) > 0.10 else "No"

    f1 = f1_score(y_true, y_pred, average="weighted")

    return {
        "Model": name,
        "Train Accuracy": round(train_accuracy, 4),
        "Test Accuracy": round(test_accuracy, 4),
        "Overfit": overfit,
        "F1 Score": round(f1, 4)
    }

In [ ]:
# Evaluating all models.
results = []

results.append(evaluate_model(model1, history1, "Basic CNN"))
results.append(evaluate_model(model2, history2, "VGG16"))
results.append(evaluate_model(model3, history3, "ResNet50 + Augmentation"))

In [ ]:
# Model comparison table.
results_df = pd.DataFrame(results)
print("\nModel Comparison Table: ")
display(results_df.style.background_gradient(subset = ["Train Accuracy", "Test Accuracy", "F1 Score"], cmap = "Greens").highlight_max(subset = ["Train Accuracy", "Test Accuracy", "F1 Score"], color = "darkgreen").set_caption("COVID-19 CNN Model Comparison")) # This single line displays a styled, color-formatted DataFrame table for comparing our CNN models.
# results_df.style: Accesses Pandas styling API — lets us apply visual formatting to the DataFrame without changing the data.
# .background_gradient: Applies a green color gradient to those 3 columns — lower values get light green, higher values get dark green. Makes it easy to visually spot better/worse models.
# .highlight_max(...): Highlights the single highest value in each of those 3 columns with a dark green background — instantly shows which model performed best.
# .set_caption(...): Adds a title/caption above the table.
# display(...): Renders the styled table in the notebook output — without display(), the styling won't show properly in Jupyter.

##### From the output above, we can infer that Basic CNN is the best-performing model overall. Therefore, we will save Basic CNN as our best model.

In [ ]:
# Saving the best model - Basic CNN.
best_idx = results_df["F1 Score"].idxmax() # Looks at the "F1 Score" column in results_df. Here, .idxmax() returns the index (i.e., [0]) of the maximum value.
best_name = results_df.loc[best_idx, "Model"] # Uses the above index (i.e., [0]) to look up the model name from the "Model" column. Here, .loc[0, "Model"] returns: best_name = "Basic CNN".
best_model = [model1, model2, model3][best_idx] # Creates a Python list of our 3 actual model objects. Uses best_idx (i.e., [0]) to pick the right one. Here, [model1, model2, model3][0] returns: best_model = model1 (Basic CNN).

best_model.save("COVID-19_best_model.keras")
print(f"\nBest Model: {best_name}")